## Active protocol amendment

The historical native VSB result is invalid for final interpretation because its input geometry differs from MATLAB. The active protocol keeps the CNNs, PD/NonPD target, grouped splits, OOF, five seeds, MCC and paired statistics, while using a shared-encoder multi-instance bag. MATLAB uses `K=1`; VSB candidate selection uses seed 42 and development data only before freezing v3.

# Confirmatory experimental protocol

**Objective.** Freeze all dataset-specific and shared decisions before final evaluation.

**Inputs.** The two completed data audits and the versioned confirmatory YAML configuration.

**Outputs.** Split manifests, frozen configuration and a protocol checklist.

**Experimental role.** Pre-registration boundary for all confirmatory training and testing.

**Leakage constraints.** No test-derived model, calibration, threshold, weight or feature decision; `Te2` remains closed until the freeze is written.

In [1]:
from pathlib import Path
from partial_discharge_adaptive_fusion.protocol import assert_development_selection_ready, assert_protocol_frozen, load_experiment_config

draft = load_experiment_config('configs/experiments/two-dataset-confirmatory-v1.yaml')
v1 = load_experiment_config('configs/experiments/two-dataset-confirmatory-v1-frozen.yaml')
v2 = load_experiment_config('configs/experiments/two-dataset-confirmatory-v2-batch4-localraw.yaml')
selection = load_experiment_config('configs/experiments/two-dataset-vsb-window-selection-localraw.yaml')
assert draft['datasets']['engineering-vsb-power-line-fault-detection']['input_policy']['status'] == 'unresolved'
assert_development_selection_ready(selection)
assert_protocol_frozen(v1); assert_protocol_frozen(v2)
assert v2['experts']['batch_size'] == 4 and v2['experts']['inference_batch_size'] == 4
assert v2['experts']['gradient_accumulation_steps'] == 1
v3_path = Path('configs/experiments/two-dataset-confirmatory-v3-windowed-localraw.yaml')
if v3_path.exists():
    v3 = load_experiment_config(v3_path); assert_protocol_frozen(v3); print('Frozen v3 is ready:', v3['config_version'])
else:
    print('Selection protocol is ready; run scripts/select_vsb_window_protocol.py to freeze v3.')

Draft remains preserved; v1 and the reviewed v2 batch-4 protocol are valid.


## Findings and handoff

The draft remains unresolved by design; the native v2 YAML and outputs are historical diagnostics. The v3 YAML is generated only after development-only window selection and is the only configuration eligible for revised confirmatory training. All configuration changes must be versioned and reviewed before opening locked test partitions.

**Next stage:** execute the reusable source pipeline on the remote GPU server.

## V4 protocol note — [PROPOSED · SCIENTIFIC]

The canonical runtime data source is repository-local `data/raw`, resolved through `PD_RAW_DATA_ROOT`. MATLAB retains 400-sample parent signals. VSB retains 800,000-sample parent signals, three phases per `id_measurement`, and deterministic end-aligned unpadded windows. Window labels are never independent examples; splits and supervision remain at parent-signal level. VSB development selection must not open the grouped holdout or the unlabeled official test.
